In [42]:
import pandas as pd
import numpy as np

train_file = open('./LightGCN/train.txt', 'r')
test_file = open('./LightGCN/test.txt', 'r')

uid_ls, his_ls = [], []

for train_line, test_line in zip(train_file.readlines(), test_file.readlines()):
    tmp_train_line = train_line.strip().split(' ')
    tmp_test_line = test_line.strip().split(' ')

#     uid = int(tmp_train_line[0])
    uid = int(tmp_train_line[0]) + 1 #因为是从lightGCN从0开始编码，但是我们要把0当作padding

    assert tmp_train_line[0] == tmp_test_line[0]

    his = [int(iid)+1 for iid in tmp_train_line[1:]] + \
            [int(iid)+1 for iid in tmp_test_line[1:]]
    
    uid_ls.append(uid)
    his_ls.append(his)




In [43]:
use_all_his = pd.DataFrame(
    data = zip(uid_ls, his_ls), columns=['user_id', 'user_history']
)

use_all_his

,user_id,user_history
0,1,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
1,2,"[19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 3..."
2,3,"[554, 555, 556, 557, 558, 559, 560, 561, 562, ..."
3,4,"[561, 559, 600, 555, 566, 622, 573, 578, 588, ..."
4,5,"[625, 626, 627, 628, 629, 630, 631, 632, 633, ..."
...,...,...
31663,31664,"[32941, 36662, 28423, 11153, 37681, 3207, 3536..."
31664,31665,"[11125, 7366, 16965, 21421, 16885, 24961, 1789..."
31665,31666,"[8846, 9847, 10252, 26225, 14973, 11528, 13748..."
31666,31667,"[37463, 29817, 2356, 23560, 9633, 13381, 6401,..."


In [44]:
import ast
df = use_all_his.copy()

# Flatten all lists in 'user_history' column
all_ids = [item for sublist in df['user_history'] for item in sublist]

# Get unique values and count
unique_ids = set(all_ids)
unique_ids_count = len(unique_ids)

print("Total number of unique IDs:", unique_ids_count)


Total number of unique IDs: 38048


In [45]:
num_users = df['user_id'].nunique()
print("Number of unique users:", num_users)

Number of unique users: 31668


In [46]:
use_all_his.to_csv('../../data_bert4rec/Yelp/dataset/user_with_full_his.tsv', sep='\t', index=False)

In [47]:
import numpy as np

all_user_item_inter_pd = use_all_his.copy()

all_users = all_user_item_inter_pd.shape[0]
test_val_users = np.random.choice(all_users, int(all_users * 0.2), replace=False)
test = test_val_users[:len(test_val_users)//2]
val = test_val_users[len(test_val_users)//2:]

training_data = all_user_item_inter_pd[
    ~all_user_item_inter_pd['user_id'].isin(test_val_users)
].copy()
val_data = all_user_item_inter_pd[
    all_user_item_inter_pd['user_id'].isin(val)
].copy()
test_data = all_user_item_inter_pd[
    all_user_item_inter_pd['user_id'].isin(test)
].copy()

training_data.shape, val_data.shape, test_data.shape

((25335, 2), (3167, 2), (3166, 2))

In [48]:
val_data.head()

,user_id,user_history
1,2,"[19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 3..."
9,10,"[779, 780, 781, 782, 783, 784, 16, 785, 786, 7..."
16,17,"[950, 933, 948, 951, 942, 935, 2, 941, 937, 3,..."
25,26,"[605, 1311, 567, 1312, 593, 1313, 575, 555, 56..."
40,41,"[2646, 2647, 747, 2648, 2649, 2650, 2651, 2652..."


In [49]:
val_data['given_user_history'] = val_data['user_history'].apply(
    lambda x:
        x[:int(0.8*len(x))] 
        # if len(x) >= 10
        # else x[:-2]
)
val_data['predicting_items'] = val_data['user_history'].apply(
    lambda x:
        x[int(0.8*len(x)):] 
        # if len(x) >= 10
        # else x[-2:]
)

In [50]:
test_data['given_user_history'] = test_data['user_history'].apply(
    lambda x:
        x[:int(0.8*len(x))] 
        # if len(x) >= 10
        # else x[:-2]
)
test_data['predicting_items'] = test_data['user_history'].apply(
    lambda x:
        x[int(0.8*len(x)):] 
        # if len(x) >= 10
        # else x[-2:]
)

In [51]:
val_data.head()

,user_id,user_history,given_user_history,predicting_items
1,2,"[19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 3...","[19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 3...","[24052, 32516, 21811, 32099, 2978, 22010, 2669..."
9,10,"[779, 780, 781, 782, 783, 784, 16, 785, 786, 7...","[779, 780, 781, 782, 783, 784, 16, 785, 786, 7...","[4937, 2781, 3519, 3323, 4331, 13373, 772]"
16,17,"[950, 933, 948, 951, 942, 935, 2, 941, 937, 3,...","[950, 933, 948, 951, 942, 935, 2, 941, 937, 3,...","[18424, 644, 6296, 629, 1348, 1341, 1283, 3102..."
25,26,"[605, 1311, 567, 1312, 593, 1313, 575, 555, 56...","[605, 1311, 567, 1312, 593, 1313, 575, 555, 56...","[8057, 4845, 6065, 9968, 3090, 20426]"
40,41,"[2646, 2647, 747, 2648, 2649, 2650, 2651, 2652...","[2646, 2647, 747, 2648, 2649, 2650, 2651, 2652...","[20992, 4686, 3049, 3837, 35022]"


In [52]:
val_data.drop(columns=['user_history']).to_csv(
    '../../data_bert4rec/Yelp/dataset/validation.tsv', sep='\t', index=False
)
test_data.drop(columns=['user_history']).to_csv(
    '../../data_bert4rec/Yelp/dataset/test.tsv', sep='\t', index=False
)
training_data.to_csv(
    '../../data_bert4rec/Yelp/dataset/training.tsv', sep='\t', index=False
)